# 🎬 AutoShorts AI — Fábrica Automatizada de Vídeos 3D (100% no Colab, Sem API Key!)

Tudo roda **direto dentro da máquina do Google Colab** usando apenas **login por link nas suas contas**:
- **Seu Próprio Roteiro (Célula 6A):** cole o seu roteiro certinho com `N:` (Narrador), `C:` (Co-protagonista) e opcionalmente `S:` (3º personagem) — o vídeo usa **exatamente as suas palavras**, sem alterar nada!
- **Google Drive:** login oficial por link/popup (`drive.mount`).
- **OpenCode CLI:** instalado direto no Colab + login por link (`opencode auth login`) para gerar roteiros automáticos ou revisar código.
- **Vozes PT-BR (`edge-tts`), Alinhamento, Legendas, Motor 3D Three.js, Playwright Chromium, FFmpeg, SFX, Música e QA:** rodam 100% locais no Colab sem precisar de nenhuma API key!

### Antes de começar
No menu **Ambiente de execução → Alterar tipo de ambiente de execução**, selecione **GPU T4** (se disponível) e clique em **Salvar**.

In [ ]:
#@title 1. Verificar Ambiente do Colab (CPU, RAM, GPU CUDA, FFmpeg, Node.js)
import os, platform, shutil, subprocess

print(f"Sistema: {platform.system()} {platform.release()} | Python: {platform.python_version()}")
print(f"CPUs lógicas: {os.cpu_count()}")

for cmd in ("nvidia-smi", "ffmpeg", "ffprobe", "node", "npm", "opencode"):
    print(f"  - {cmd:12s}: {shutil.which(cmd) or 'ainda não instalado'}")

if shutil.which("nvidia-smi"):
    print("\n--- GPU NVIDIA alocada ---")
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], text=True))
else:
    print("\n⚠️ Nenhuma GPU NVIDIA alocada neste runtime (modo CPU).")

In [ ]:
#@title 2. Clonar do GitHub e Instalar Tudo no Colab (Python + Node + Chromium + FFmpeg + OpenCode CLI)
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/99X123/IA-geradora-de-video-roblox.git"  #@param {type:"string"}
PROJECT_DIR = Path("/content/AutoShortsAI")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"Atualizando repositório existente em {PROJECT_DIR}...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)

# 1. FFmpeg e bibliotecas do Chromium
!apt-get -qq update && apt-get -qq install -y ffmpeg libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2t64 || apt-get -qq install -y ffmpeg

# 2. Dependências Python
!pip install -q -r requirements-colab.txt

# 3. Dependências Node.js (Three.js + Playwright Chromium)
!npm install --no-audit --no-fund
!npx playwright install chromium

# 4. Instalar OpenCode CLI direto na máquina do Colab
!curl -fsSL https://opencode.ai/install | bash
for bin_dir in (os.path.expanduser("~/.opencode/bin"), os.path.expanduser("~/.local/bin")):
    if os.path.isdir(bin_dir) and bin_dir not in os.environ.get("PATH", ""):
        os.environ["PATH"] = bin_dir + ":" + os.environ.get("PATH", "")

!opencode --version || echo "OpenCode será ativado após o PATH"
print("\n✅ Tudo instalado direto na máquina do Colab!")

In [ ]:
#@title 3. Teste Real de GPU / WebGL no Chromium (Hardware vs SwiftShader)
import json, os, sys
sys.path.insert(0, os.getcwd())

from src.video_orchestrator import probe_render_environment

env_probe = probe_render_environment()
print(json.dumps(env_probe, indent=2, ensure_ascii=False))

if env_probe["gpu_webgl_confirmed"]:
    print("\n✅ WebGL acelerado por HARDWARE confirmado!")
else:
    print("\n⚠️ AVISO: Chromium está usando SOFTWARE (SwiftShader/fallback seguro).")
    print("   A renderização funciona normalmente; use qualidade 'draft' para testes rápidos ou 'production' para o vídeo final.")

In [ ]:
#@title 4. Login no Google Drive (por link oficial do Colab) e Criar Pastas
from src.drive_manager import DriveManager

MONTAR_GOOGLE_DRIVE = True  #@param {type:"boolean"}

if MONTAR_GOOGLE_DRIVE:
    dm = DriveManager()
    drive_base = dm.mount_if_colab()
    print(f"✅ Google Drive conectado em: {drive_base}")
    for sub in sorted(drive_base.iterdir()):
        print("  📁", sub.name)
else:
    print("Google Drive desativado nesta execução.")

In [ ]:
#@title 5. Login na Conta do OpenCode por Link (Opcional — Sem API Key!)
#@markdown Marque `FAZER_LOGIN_OPENCODE` para abrir o link oficial de login na sua conta.
#@markdown Se você for usar o SEU PRÓPRIO ROTEIRO na Célula 6A, nem precisa logar aqui!
import os

FAZER_LOGIN_OPENCODE = False  #@param {type:"boolean"}
PROVEDOR_LOGIN = ""  #@param ["", "google", "opencode", "github-copilot", "openai", "anthropic"]

if FAZER_LOGIN_OPENCODE:
    if PROVEDOR_LOGIN:
        !opencode auth login {PROVEDOR_LOGIN}
    else:
        !opencode auth login

print("=== Contas conectadas no OpenCode ===")
!opencode auth list || true

print("\n=== Modelos disponíveis nesta máquina do Colab ===")
!opencode models | head -n 35 || true

In [ ]:
#@title 6A. Cole ou Escreva o SEU PRÓPRIO ROTEIRO Certinho Aqui! ✍️
#@markdown - Deixe `USAR_MEU_ROTEIRO = True` para o vídeo usar **exatamente o texto abaixo**, sem a IA inventar falas.
#@markdown - Use `N:` para o Narrador (voz 1), `C:` para o Co-protagonista (voz 2) e `S:` para um 3º personagem opcional (voz 3).
#@markdown - Opcional: coloque o cenário antes da fala entre colchetes: `[sala]`, `[sala_noite]`, `[cozinha]`, `[corredor]`, `[praca]`, `[praca_dourada]`, `[praca_caos]`, `[estudio]`.
#@markdown - Se preferir que a IA gere o roteiro pelo tema, basta desmarcar `USAR_MEU_ROTEIRO`.

USAR_MEU_ROTEIRO = True  #@param {type:"boolean"}
TITULO_DO_VIDEO = "Meu Robô na Cozinha"  #@param {type:"string"}
PERSONAGEM_COSTAR = "bot"  #@param ["auto", "bot", "pigeon", "phantom", "rock", "human", "mech"]

MEU_ROTEIRO = """
[sala] N: Eu só queria dormir cedo hoje, até meu robô aparecer com uma panela gigante.
[cozinha] C: Calma Leo! Isso aqui vai virar o melhor brigadeiro da galáxia.
[cozinha] N: Spoiler: em três segundos a cozinha inteira ficou cheia de espuma brilhante.
[praca] C: Olha pelo lado positivo, todo mundo na praça finalmente percebeu nosso talento!
[sala_noite] N: Alguém sabe como mudar de planeta antes de segunda-feira?
"""

if USAR_MEU_ROTEIRO:
    linhas_validas = [l.strip() for l in MEU_ROTEIRO.strip().splitlines() if l.strip() and not l.strip().startswith("#")]
    print(f"✅ SEU ROTEIRO ATIVADO! ({len(linhas_validas)} falas serão narradas exatamente como você escreveu)")
    for i, l in enumerate(linhas_validas, 1):
        print(f"  {i:02d}. {l}")
else:
    print("ℹ️ Modo automático ativado: a história será gerada a partir do TEMA na Célula 6B.")

In [ ]:
#@title 6B. Configuração de Vídeo, Voz e Qualidade
TEMA = "meu robô assistente resolveu fazer café e ativou o modo turbo"  #@param {type:"string"}
DURACAO_SEGUNDOS = 60  #@param {type:"slider", min:12, max:90, step:6}
QUALIDADE = "production"  #@param ["draft", "preview", "production", "ultra"]
NUM_PERSONAGENS = 2  #@param {type:"slider", min:2, max:3, step:1}
PROVEDOR_ROTEIRO = "opencode"  #@param ["opencode", "local_template", "auto"]
MODELO_OPENCODE = ""  #@param {type:"string"}
PROVEDOR_TTS = "edge_tts"  #@param ["edge_tts", "auto", "local_synth_test"]
ATIVAR_SFX = True  #@param {type:"boolean"}
ATIVAR_MUSICA = True  #@param {type:"boolean"}
PAUSAR_APROVACAO_STORYBOARD = False  #@param {type:"boolean"}
APROVAR_STORYBOARD = False  #@param {type:"boolean"}
WORKERS = 2  #@param {type:"slider", min:1, max:4, step:1}
NUM_PARTES = 10  #@param {type:"slider", min:2, max:16, step:1}
SALVAR_NO_DRIVE = True  #@param {type:"boolean"}
PROJECT_ID_RETOMAR = ""  #@param {type:"string"}

from src.config import AutoShortsConfig

config = AutoShortsConfig.load(
    overrides={
        "video": {
            "quality_preset": QUALIDADE,
            "target_duration_sec": DURACAO_SEGUNDOS,
            "workers": WORKERS,
            "num_parts": NUM_PARTES,
        },
        "story": {
            "num_characters": NUM_PERSONAGENS,
            "provider": PROVEDOR_ROTEIRO,
            "require_storyboard_approval": PAUSAR_APROVACAO_STORYBOARD,
        },
        "tts": {"provider": PROVEDOR_TTS},
        "audio": {"enable_sfx": ATIVAR_SFX, "enable_music": ATIVAR_MUSICA},
        "opencode": {"model": MODELO_OPENCODE.strip()},
        "drive": {"enabled": SALVAR_NO_DRIVE and MONTAR_GOOGLE_DRIVE},
    }
)
modo_txt = "ROTEIRO PRÓPRIO (Célula 6A)" if USAR_MEU_ROTEIRO else f"AUTOMÁTICO ({PROVEDOR_ROTEIRO})"
print(f"Configuração pronta: {config.get('video.width')}x{config.get('video.height')} @ {config.get('video.fps')}fps | Modo: {modo_txt} | TTS: {PROVEDOR_TTS}")

In [ ]:
#@title 7. Produzir Vídeo Completo (Roda tudo no Colab -> QA -> Salva no Google Drive)
import base64
from pathlib import Path
from IPython.display import HTML, Image as IPyImage, display
from src.video_orchestrator import VideoOrchestrator

WORK_ROOT = Path("/content/autoshorts_work")
orch = VideoOrchestrator(config=config, work_root=WORK_ROOT)

tema_final = TITULO_DO_VIDEO.strip() if (USAR_MEU_ROTEIRO and TITULO_DO_VIDEO.strip()) else TEMA
roteiro_usuario = MEU_ROTEIRO if USAR_MEU_ROTEIRO else None

result = orch.produce(
    theme=tema_final,
    project_id=PROJECT_ID_RETOMAR.strip() or None,
    duration_sec=DURACAO_SEGUNDOS,
    custom_script=roteiro_usuario,
    costar_rig=PERSONAGEM_COSTAR,
    approved=APROVAR_STORYBOARD,
    enable_drive_sync=SALVAR_NO_DRIVE and MONTAR_GOOGLE_DRIVE,
)

print(f"\nStatus: {result['status']} | Project ID: {result['project_id']}")

if result["status"] == "awaiting_storyboard_approval":
    print("⏸️ Pausado para aprovação do storyboard. Defina PROJECT_ID_RETOMAR =", repr(result["project_id"]), "e APROVAR_STORYBOARD = True.")
    display(IPyImage(filename=result["storyboard_preview"]["contact_sheet"]))
else:
    qa = result["quality_report"]
    print(f"QA aprovado: {qa['approved']} | Técnicos: {qa['technical_passed']} | Falhas: {qa['failed_checks']}")
    print(f"Alinhamento: {result['alignment_method']} (fallback={result['alignment_fallback_used']})")
    if qa["visual_checks"].get("contact_sheet"):
        print("\n--- Folha de Contato (QA Visual) ---")
        display(IPyImage(filename=qa["visual_checks"]["contact_sheet"]))

    mp4_path = Path(result["video_mp4"])
    if mp4_path.exists() and mp4_path.stat().st_size < 45 * 1024 * 1024:
        b64 = base64.b64encode(mp4_path.read_bytes()).decode("ascii")
        display(HTML(f'<video controls width="360" src="data:video/mp4;base64,{b64}"></video>'))
    if result.get("drive_sync"):
        print("\n☁️ Salvo no Google Drive:", result["drive_sync"]["video_drive_path"])

In [ ]:
#@title 8. (Opcional) Re-renderizar Apenas Partes Específicas com Erro
PARTES_PARA_RERENDER = ""  #@param {type:"string"}

if PARTES_PARA_RERENDER.strip() and "project_dir" in result:
    idxs = [int(x.strip()) for x in PARTES_PARA_RERENDER.split(",") if x.strip().isdigit()]
    print(f"Re-renderizando apenas as partes: {idxs}...")
    orch.rerender_scene_parts(Path(result["project_dir"]), idxs)
    print("✅ Re-renderização parcial concluída!")
else:
    print("Nenhuma parte selecionada (ex.: informe '2,5' para re-renderizar apenas as partes 2 e 5).")

In [ ]:
#@title 9. Testes Automatizados e Revisão Independente pelo OpenCode (Rodando no Colab)
import json
from src.opencode_reviewer import OpenCodeReviewer

print("=== 1. Executando suíte pytest ===")
!python -m pytest -v

EXECUTAR_REVISAO_OPENCODE = False  #@param {type:"boolean"}
PERMITIR_EDICOES_OPENCODE = False  #@param {type:"boolean"}

if EXECUTAR_REVISAO_OPENCODE:
    reviewer = OpenCodeReviewer(config=config)
    oc_report = reviewer.review_project(allow_edits=PERMITIR_EDICOES_OPENCODE)
    print("\n=== Relatório OpenCode ===")
    print(json.dumps({k: v for k, v in oc_report.items() if k != "opencode_output"}, indent=2, ensure_ascii=False))
    if oc_report.get("opencode_output"):
        print("\n--- Saída do OpenCode ---")
        print(oc_report["opencode_output"][:4000])